In [190]:
import numpy as np
import pandas as pd

In [191]:
df = pd.read_excel('c:\\Users\\U440845\\Documents\\GitHub\\Miles_Forecasting\\dataset\\Final_KPIs.xlsx', na_values='None')

In [192]:
df.head()

,month,flown miles,rpms,distinct prem (GS + 1K),accrued miles,gross passenger revenue,accrued:redeemed
0,2019-01-01,4730717361,17525531459,22992,24509761414,2.835882e+09,5
1,2019-02-01,9212110701,15576416271,32358,21734038032,2.723044e+09,2
2,2019-03-01,16371796910,20124160307,44459,25208125623,3.167433e+09,2
3,2019-04-01,16582967373,19601173274,42422,26265097065,3.068168e+09,2
4,2019-05-01,19900973419,21003785305,46735,27683273666,3.201879e+09,1


In [193]:
df["flown miles lag_3"]  = df["flown miles"].shift(3)
df["flown miles lag_12"] = df["flown miles"].shift(12)
df["flown miles rolling mean 3"] = df["flown miles"].rolling(window=3).mean()

In [194]:
df['month_index'] = df['month'].dt.month
df["year"] = df["month"].dt.year
df['month_sin'] = np.sin(2 * np.pi * df['month_index'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month_index'] / 12)

In [195]:
start_year = df['month'].dt.year.min()
start_month = df['month'].dt.month.min()
df['time_step'] = ((df['month'].dt.year - start_year) * 12 + (df['month'].dt.month - start_month))


In [196]:
df['flown miles yoy growth'] = ((df['flown miles'] - df['flown miles lag_12']) / df['flown miles lag_12'])
df.loc[df['time_step'] < 12, 'flown miles yoy growth'] = np.nan
df['flown miles yoy growth'] = df['flown miles yoy growth'].clip(-1, 2)

In [197]:
month_yoy_avg = ( df.groupby("month_index")["flown miles yoy growth"].mean().rename("month_yoy_avg_growth"))
df = df.join(month_yoy_avg, on="month_index")

In [198]:
df["log_flown_miles"] = np.log1p(df["flown miles"])

In [199]:
df.head()

,month,flown miles,rpms,distinct prem (GS + 1K),accrued miles,gross passenger revenue,accrued:redeemed,flown miles lag_3,flown miles lag_12,flown miles rolling mean 3,month_index,year,month_sin,month_cos,time_step,flown miles yoy growth,month_yoy_avg_growth,log_flown_miles
0,2019-01-01,4730717361,17525531459,22992,24509761414,2.835882e+09,5,NaN,NaN,NaN,1,2019,0.500000,8.660254e-01,0,NaN,0.654319,22.277343
1,2019-02-01,9212110701,15576416271,32358,21734038032,2.723044e+09,2,NaN,NaN,NaN,2,2019,0.866025,5.000000e-01,1,NaN,0.457455,22.943785
2,2019-03-01,16371796910,20124160307,44459,25208125623,3.167433e+09,2,NaN,NaN,1.010487e+10,3,2019,1.000000,6.123234e-17,2,NaN,0.253648,23.518826
3,2019-04-01,16582967373,19601173274,42422,26265097065,3.068168e+09,2,4.730717e+09,NaN,1.405562e+10,4,2019,0.866025,-5.000000e-01,3,NaN,0.498118,23.531642
4,2019-05-01,19900973419,21003785305,46735,27683273666,3.201879e+09,1,9.212111e+09,NaN,1.761858e+10,5,2019,0.500000,-8.660254e-01,4,NaN,0.448148,23.714034


In [200]:
cols_to_drop = [
    "rpms",
    "month_index",
    "flown miles yoy growth"]
df.drop(columns=cols_to_drop, inplace=True)
df.head()

,month,flown miles,distinct prem (GS + 1K),accrued miles,gross passenger revenue,accrued:redeemed,flown miles lag_3,flown miles lag_12,flown miles rolling mean 3,year,month_sin,month_cos,time_step,month_yoy_avg_growth,log_flown_miles
0,2019-01-01,4730717361,22992,24509761414,2.835882e+09,5,NaN,NaN,NaN,2019,0.500000,8.660254e-01,0,0.654319,22.277343
1,2019-02-01,9212110701,32358,21734038032,2.723044e+09,2,NaN,NaN,NaN,2019,0.866025,5.000000e-01,1,0.457455,22.943785
2,2019-03-01,16371796910,44459,25208125623,3.167433e+09,2,NaN,NaN,1.010487e+10,2019,1.000000,6.123234e-17,2,0.253648,23.518826
3,2019-04-01,16582967373,42422,26265097065,3.068168e+09,2,4.730717e+09,NaN,1.405562e+10,2019,0.866025,-5.000000e-01,3,0.498118,23.531642
4,2019-05-01,19900973419,46735,27683273666,3.201879e+09,1,9.212111e+09,NaN,1.761858e+10,2019,0.500000,-8.660254e-01,4,0.448148,23.714034


In [201]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [202]:
TARGET = "flown miles"
FEATURES = ["distinct prem (GS + 1K)",
    "accrued miles",
    "gross passenger revenue",
    "accrued:redeemed",
    "flown miles lag_3",
    "flown miles lag_12", 
    "flown miles rolling mean 3",
    "month_sin",
    "month_cos",
    "time_step",
    "month_yoy_avg_growth"]

In [203]:
train_df = df[df["year"] <= 2023]
val_df   = df[df["year"] == 2024]
test_df  = df[df["year"] == 2025]

In [204]:
X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val     = val_df[FEATURES], val_df[TARGET]
X_test, y_test   = test_df[FEATURES], test_df[TARGET]

In [205]:
model = XGBRegressor(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.03,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=2.0,
    objective="reg:squarederror",
    random_state=42
)

In [206]:
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [207]:
test_preds = model.predict(X_test)

mae = mean_absolute_error(y_test, test_preds)
rmse = np.sqrt(mean_squared_error(y_test, test_preds))
mape = np.mean(np.abs((y_test - test_preds) / y_test)) * 100

print(f"Test MAE:  {mae:,.0f}")
print(f"Test RMSE: {rmse:,.0f}")
print(f"Test MAPE: {mape:.2f}%")

Test MAE:  3,000,093,440
Test RMSE: 3,905,885,359
Test MAPE: 9.16%


In [208]:
final_train_df = df[df["year"] <= 2025]

X_final = final_train_df[FEATURES]
y_final = final_train_df[TARGET]

final_model = XGBRegressor(
    n_estimators=model.n_estimators,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

final_model.fit(X_final, y_final)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [209]:
future_months = pd.date_range(
    start="2026-01-01",
    end="2026-12-01",
    freq="MS"
)

df_2026 = pd.DataFrame({"month": future_months})
df_2026["year"] = df_2026["month"].dt.year
df_2026["month_index"] = df_2026["month"].dt.month

In [210]:
df_2026["month_sin"] = np.sin(2 * np.pi * df_2026["month_index"] / 12)
df_2026["month_cos"] = np.cos(2 * np.pi * df_2026["month_index"] / 12)

df_2026["time_step"] = np.arange(
    df["time_step"].max() + 1,
    df["time_step"].max() + 1 + len(df_2026)
)

In [211]:
last = df.iloc[-1]

df_2026["flown miles lag_3"]  = last["flown miles rolling mean 3"]
df_2026["flown miles lag_12"] = last["flown miles lag_12"]
df_2026["flown miles rolling mean 3"] = last["flown miles rolling mean 3"]

In [212]:
for col in [
    "distinct prem (GS + 1K)",
    "accrued miles",
    "gross passenger revenue",
    "accrued:redeemed",
    "month_yoy_avg_growth"
]:
    df_2026[col] = last[col]

In [213]:
df_2026["forecast_flown_miles"] = final_model.predict(
    df_2026[FEATURES]
)

In [214]:
df_2026[["month", "forecast_flown_miles"]]

,month,forecast_flown_miles
0,2026-01-01,3.881086e+10
1,2026-02-01,3.884519e+10
2,2026-03-01,3.887560e+10
3,2026-04-01,3.890149e+10
4,2026-05-01,3.910654e+10
5,2026-06-01,3.932348e+10
6,2026-07-01,3.911813e+10
7,2026-08-01,3.858548e+10
8,2026-09-01,3.856271e+10
9,2026-10-01,3.852529e+10
